# Rpy2

In [49]:
from rpy2.robjects.packages import importr, data
from pymer4.bridge import R2pandas, R2numpy
import rpy2.robjects as ro

In [2]:
base = importr('base')
utils = importr('utils')
stats = importr('stats')

In [69]:
lme4 = importr('lme4')
lmerTest = importr('lmerTest')
broom = importr('broom')
broomm = importr('broom.mixed')
emmeans = importr('emmeans')

# Namespacing
summary = base.summary
lmer = lmerTest.lmer
lm = stats.lm
joint_tests = emmeans.joint_tests

## Broom

In [82]:
def tidy(model):

    if isinstance(model, ro.methods.RS4):
        func = broomm.tidy_merMod
    if isinstance(model, ro.vectors.ListVector):
        func = broom.tidy_lm

    return R2pandas(func(model))

def glance(model):

    if isinstance(model, ro.methods.RS4):
        func = broomm.glance_merMod
    if isinstance(model, ro.vectors.ListVector):
        func = broom.glance_lm

    return R2pandas(func(model))

def augment(model):

    if isinstance(model, ro.methods.RS4):
        func = broomm.augment_merMod
    if isinstance(model, ro.vectors.ListVector):
        func = broom.augment_lm

    return R2pandas(func(model))

def anova(*models):
    return R2pandas(stats.anova(*models))

In [88]:
def santize_column_names(df):
    new_cols = []
    for col in df.columns:
        col = col.strip('')
        col = col.replace('.', '_')
        col = col.lstrip('_')
        new_cols.append(col)
    df.columns = new_cols
    return df

In [42]:
sleepstudy = data(lme4).fetch('sleepstudy')['sleepstudy']

In [59]:
m_lm = lm('Reaction ~ Days', data=sleepstudy)
m_lmm = lmer('Reaction ~ Days + (1|Subject)',data=sleepstudy)

In [61]:
tidy(m_lm)
tidy(m_lmm)

,term,estimate,std.error,statistic,p.value
1,(Intercept),251.405105,6.610154,38.033169,2.156888e-87
2,Days,10.467286,1.238195,8.453663,9.894096e-15


,effect,group,term,estimate,std.error,statistic,df,p.value
1,fixed,NA_character_,(Intercept),251.405105,9.746716,25.793826,22.810199,2.241351e-18
2,fixed,NA_character_,Days,10.467286,0.804221,13.015428,161.000000,6.412601e-27
3,ran_pars,Subject,sd__(Intercept),37.123827,NaN,NaN,NaN,NaN
4,ran_pars,Residual,sd__Observation,30.991234,NaN,NaN,NaN,NaN


In [71]:
anova(m_lm)

,Df,Sum Sq,Mean Sq,F value,Pr(>F)
Days,1,162702.65191,162702.65191,71.464421,9.894096e-15
Residuals,178,405251.61748,2276.69448,NaN,NaN


In [72]:
anova(m_lmm)

,Sum Sq,Mean Sq,NumDF,DenDF,F value,Pr(>F)
Days,162702.65191,162702.65191,1,161.0,169.401361,6.412601e-27


In [83]:
anova(m_lmm, m_lm)

R[write to console]: refitting model(s) with ML (instead of REML)



,npar,AIC,BIC,logLik,deviance,Chisq,Df,Pr(>Chisq)
MODEL2,3.0,1906.293056,1915.871927,-950.146528,1900.293056,NaN,NaN,NaN
MODEL1,4.0,1802.078643,1814.850470,-897.039322,1794.078643,106.214413,1.0,6.617359e-25


In [89]:
santize_column_names(glance(m_lm))

,r_squared,adj_r_squared,sigma,statistic,p_value,df,logLik,AIC,BIC,deviance,df_residual,nobs
1,0.286471,0.282463,47.71472,71.464421,9.894096e-15,1.0,-950.146528,1906.293056,1915.871927,405251.61748,178,180


In [64]:
glance(m_lm)
glance(m_lmm)

,r.squared,adj.r.squared,sigma,statistic,p.value,df,logLik,AIC,BIC,deviance,df.residual,nobs
1,0.286471,0.282463,47.71472,71.464421,9.894096e-15,1.0,-950.146528,1906.293056,1915.871927,405251.61748,178,180


,nobs,sigma,logLik,AIC,BIC,REMLcrit,df.residual
1,180,30.991234,-893.232543,1794.465085,1807.236913,1786.465085,176


In [68]:
augment(m_lm).head()
augment(m_lmm).head()

,Reaction,Days,.fitted,.resid,.hat,.sigma,.cooksd,.std.resid
1,249.5600,0.0,251.405105,-1.845105,0.019192,47.849112,0.000015,-0.039046
2,258.7047,1.0,261.872391,-3.167691,0.013805,47.848717,0.000031,-0.066851
3,250.8006,2.0,272.339677,-21.539077,0.009764,47.821650,0.001015,-0.453634
4,321.4398,3.0,282.806963,38.632837,0.007071,47.760496,0.002351,0.812541
5,356.8519,4.0,293.274249,63.577651,0.005724,47.608706,0.005140,1.336283


,Reaction,Days,Subject,.fitted,.resid,.hat,.cooksd,.fixed,.mu,.offset,.sqrtXwt,.sqrtrwt,.weights,.wtres
1,249.5600,0.0,308,292.188815,-42.628815,0.107483,0.127646,251.405105,292.188815,0.0,1.0,1.0,1.0,-42.628815
2,258.7047,1.0,308,302.656101,-43.951401,0.102096,0.127347,261.872391,302.656101,0.0,1.0,1.0,1.0,-43.951401
3,250.8006,2.0,308,313.123387,-62.322787,0.098056,0.243725,272.339677,313.123387,0.0,1.0,1.0,1.0,-62.322787
4,321.4398,3.0,308,323.590673,-2.150873,0.095362,0.000281,282.806963,323.590673,0.0,1.0,1.0,1.0,-2.150873
5,356.8519,4.0,308,334.057959,22.793941,0.094015,0.030980,293.274249,334.057959,0.0,1.0,1.0,1.0,22.793941


In [90]:
santize_column_names(augment(m_lmm))

,Reaction,Days,Subject,fitted,resid,hat,cooksd,fixed,mu,offset,sqrtXwt,sqrtrwt,weights,wtres
1,249.5600,0.0,308,292.188815,-42.628815,0.107483,0.127646,251.405105,292.188815,0.0,1.0,1.0,1.0,-42.628815
2,258.7047,1.0,308,302.656101,-43.951401,0.102096,0.127347,261.872391,302.656101,0.0,1.0,1.0,1.0,-43.951401
3,250.8006,2.0,308,313.123387,-62.322787,0.098056,0.243725,272.339677,313.123387,0.0,1.0,1.0,1.0,-62.322787
4,321.4398,3.0,308,323.590673,-2.150873,0.095362,0.000281,282.806963,323.590673,0.0,1.0,1.0,1.0,-2.150873
5,356.8519,4.0,308,334.057959,22.793941,0.094015,0.030980,293.274249,334.057959,0.0,1.0,1.0,1.0,22.793941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,329.6076,5.0,372,321.857281,7.750319,0.094015,0.003582,303.741535,321.857281,0.0,1.0,1.0,1.0,7.750319
177,334.4818,6.0,372,332.324567,2.157233,0.095362,0.000282,314.208821,332.324567,0.0,1.0,1.0,1.0,2.157233
178,343.2199,7.0,372,342.791853,0.428047,0.098056,0.000011,324.676107,342.791853,0.0,1.0,1.0,1.0,0.428047
179,369.1417,8.0,372,353.259139,15.882561,0.102096,0.016630,335.143393,353.259139,0.0,1.0,1.0,1.0,15.882561


## Load dataset function

In [9]:
from pymer4.io import load_dataset

sleep = load_dataset('sleepstudy')
sleep.head()

,Reaction,Days,Subject
1,249.5600,0.0,308
2,258.7047,1.0,308
3,250.8006,2.0,308
4,321.4398,3.0,308
5,356.8519,4.0,308


In [138]:
import rpy2.robjects as ro
import rpy2.robjects.vectors as rv
import rpy2.robjects.packages as rpackages
import pandas as pd
import numpy as np


class RObject:
    """A generic Python representation of an R object, allowing attribute-style access."""

    def __init__(self, **entries):
        sanitized = {}
        for k, v in entries.items():
            sanitized[k.replace('.', "_")] = v

        self.__dict__.update(sanitized)

    def __repr__(self):
        return f"RObject({', '.join(self.__dict__.keys())})"


def r_to_python(obj):
    """
    Recursively converts an R object (from rpy2) into a native Python object.

    - R ListVector → Python object with attributes
    - R Matrix/DataFrame → Pandas DataFrame
    - R Vector → Python list (or scalar if length 1)
    """

    if obj is None or obj == ro.r("NULL") or isinstance(obj, type(ro.r("NULL"))):
        return None

    # if obj.names is None or obj.names == ro.r("NULL") or isinstance(obj.names, type(ro.r("NULL"))):
    #     return None

    # If it's an R ListVector, convert to an object with attributes
    if isinstance(obj, ro.ListVector):
        return RObject(**{name: r_to_python(obj.rx2(name)) for name in obj.names})

    # If it's an R DataFrame, convert to a Pandas DataFrame
    elif isinstance(obj, ro.DataFrame):
        return R2pandas(obj)

    # If it's an R Matrix, convert to a Pandas DataFrame
    elif isinstance(obj, rv.Matrix):
        return R2numpy(obj)

    # If it's an R Vector (e.g., numeric, integer, character), convert to Python list or scalar
    elif isinstance(obj, rv.Vector):
        values = list(obj)
        return (
            values[0] if len(values) == 1 else values
        )  # Convert to scalar if single element

    # If it's None, return None
    elif obj is None:
        return None

    # Otherwise, return as-is (fallback case)
    else:
        return obj


In [142]:
import rpy2.robjects as ro
import pandas as pd
import numpy as np

def convert_lmer_summary(lmer_model):
    """
    Convert the output of summary() for an lmer model into intuitive Python objects.
    
    Arguments:
    - lmer_model: An lmer model fitted using lme4::lmer or lmerTest::lmer.

    Returns:
    - A dictionary with:
        - 'fixed_effects': Pandas DataFrame of fixed effect estimates.
        - 'random_effects': Pandas DataFrame of random effect variance components.
        - 'model_fit': Dictionary with AIC, BIC, log-likelihood, etc.
        - 'residuals': Residual standard deviation.
        - 'degrees_of_freedom': Degrees of freedom.
    """
    # Get summary of the lmer model
    summary_r = ro.r(f"summary({lmer_model.r_repr()})")

    # Extract fixed effects coefficients
    fixed_effects = ro.r(f"as.data.frame({lmer_model.r_repr()}@beta)")
    fixed_effects.columns = ["Estimate"]  # Set column name
    
    # Extract standard errors, t-values, and p-values for fixed effects
    coef_matrix = np.array(summary_r.rx2("coefficients"))
    fixed_effects[["Std. Error", "t value", "Pr(>|t|)"]] = coef_matrix[:, 1:]

    # Extract random effects (variance components)
    random_effects = ro.r(f"as.data.frame(VarCorr({lmer_model.r_repr()}))")
    random_effects.columns = ["Group", "Effect", "Variance", "Std.Dev"]

    # Extract model fit statistics
    model_fit = {
        "AIC": float(ro.r(f"AIC({lmer_model.r_repr()})")[0]),
        "BIC": float(ro.r(f"BIC({lmer_model.r_repr()})")[0]),
        "logLik": float(ro.r(f"logLik({lmer_model.r_repr()})")[0]),
        "deviance": float(ro.r(f"deviance({lmer_model.r_repr()})")[0]),
        "REMLcrit": float(ro.r(f"{lmer_model.r_repr()}@devcomp$cmp['REML']")[0])
    }

    # Extract residual standard deviation
    residuals = float(ro.r(f"{lmer_model.r_repr()}@sigma")[0])

    # Extract degrees of freedom
    df = list(ro.r(f"{lmer_model.r_repr()}@dims['p']"))  # Extract df as list

    return {
        "fixed_effects": fixed_effects,
        "random_effects": random_effects,
        "model_fit": model_fit,
        "residuals": residuals,
        "degrees_of_freedom": df
    }


In [108]:
# this is a ListVector and I don't know what to do with it
out = summary(model)

In [136]:
out = r_to_python(out)
out

RObject(methTitle, objClass, devcomp, isLmer, useScale, logLik, family, link, ngrps, coefficients, sigma, vcov, varcor, AICtab, call, residuals, fitMsgs, optinfo, corrSet)

In [137]:
out.call

In [ ]:
out = summary(lmm)
print(out)

In [149]:
model = ro.r('lmer(Reaction ~ Days + (1|Subject), data=sleepstudy)')

In [155]:
result = ro.r('summary')(model)

In [165]:
x = 10


In [162]:
y = f"hello {x}"

In [166]:
y

'hello 11'

In [163]:
ro.r(f'summary({model.r_repr()})')

RParsingError: Parsing status not OK - PARSING_STATUS.PARSE_ERROR

In [ ]:
convert_lmer_summary(lmm)

In [99]:
[name for name in out.names]

['methTitle',
 'objClass',
 'devcomp',
 'isLmer',
 'useScale',
 'logLik',
 'family',
 'link',
 'ngrps',
 'coefficients',
 'sigma',
 'vcov',
 'varcor',
 'AICtab',
 'call',
 'residuals',
 'fitMsgs',
 'optinfo',
 'corrSet']

In [106]:
out.rx2('AICtab')

1786.465085


AttributeError: 'ListVector' object has no attribute 'keys'

In [66]:
res = r_to_python(summary(lmm))

TypeError: 'NULLType' object is not iterable

In [52]:
print(out)


RObject(call, terms, residuals, coefficients, aliased, sigma, df, r_squared, adj_r_squared, fstatistic, cov_unscaled)
